In [1]:
import os
import wave
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

# ==========================================
# 1. EXACT MICROPYTHON FUNCTIONS (COPIED FROM BOARD)
# ==========================================
def calculate_energy(buffer, start=0, end=None):
    if end is None:
        end = len(buffer)
    sum_squares = 0.0
    # Process exactly as 16-bit PCM bytes
    for i in range(start, end, 2):
        val = buffer[i] | (buffer[i+1] << 8)
        if val >= 32768: val -= 65536
        sum_squares += val * val
    num_samples = (end - start) // 2
    if num_samples == 0: return 0
    return (sum_squares / num_samples) ** 0.5

def get_trim_indices(raw_buffer, noise_floor):
    window_size = 512 
    buffer_len = len(raw_buffer)
    start_idx = 0
    end_idx = buffer_len
    
    for i in range(0, buffer_len, window_size):
        chunk_end = min(i + window_size, buffer_len)
        if calculate_energy(raw_buffer, i, chunk_end) > noise_floor * 1.5:
            start_idx = max(0, i - (window_size * 2))
            break
            
    for i in range(buffer_len - window_size, -1, -window_size):
        chunk_end = min(i + window_size, buffer_len)
        if calculate_energy(raw_buffer, i, chunk_end) > noise_floor * 1.5:
            end_idx = min(buffer_len, chunk_end + (window_size * 2))
            break
            
    if start_idx >= end_idx - 1000:
        return 0, buffer_len
    return start_idx, end_idx

# ==========================================
# SUPER-ROBUST TIME-DOMAIN FEATURE EXTRACTOR
# ==========================================
def compute_smart_features(raw_buffer, start_byte, end_byte, num_coeffs=13):
    num_samples = (end_byte - start_byte) // 2
    if num_samples <= 0: return [0.0] * num_coeffs
    
    max_amp = 1
    sum_amp = 0
    sum_high_freq = 0
    sum_low_freq = 0
    zcr = 0
    peaks = 0
    
    last_val = 0
    last_low = 0
    last_sign = 0
    last_diff_sign = 0
    
    envelope = [0.0] * 8
    samples_per_bin = max(1, num_samples // 8)
    
    sample_idx = 0
    
    for i in range(start_byte, end_byte, 2):
        val = raw_buffer[i] | (raw_buffer[i+1] << 8)
        if val >= 32768: val -= 65536
        abs_val = abs(val)
        
        # 1. Volume & Energy
        if abs_val > max_amp: max_amp = abs_val
        sum_amp += abs_val
        
        # 2. DIGITAL FILTERS (Poor-man's MFCC)
        # High-Pass Filter (Emphasizes 'S', 'F', 'T' sounds)
        diff = val - last_val
        sum_high_freq += abs(diff)
        
        # Low-Pass Filter (Emphasizes 'O', 'U', 'A' sounds)
        low_val = (last_low * 8 + val * 2) // 10
        sum_low_freq += abs(low_val)
        
        # 3. Envelope
        bin_idx = min(7, sample_idx // samples_per_bin)
        envelope[bin_idx] += abs_val
        
        # 4. Friction & Pitch (Noise Gated)
        if abs_val > 300: # Fixed noise gate
            sign = 1 if val > 0 else -1
            if sign != last_sign and last_sign != 0:
                zcr += 1
            last_sign = sign
            
            diff_sign = 1 if diff > 0 else -1 if diff < 0 else 0
            if diff_sign != last_diff_sign and last_diff_sign == 1:
                peaks += 1
            last_diff_sign = diff_sign
            
        last_val = val
        last_low = low_val
        sample_idx += 1
        
    # --- NORMALIZE FEATURES (0.0 to 1.0) ---
    f1 = (sum_amp / num_samples) / max_amp              
    f2 = sum_high_freq / (sum_amp + 1) # High Freq Ratio
    f3 = sum_low_freq / (sum_amp + 1)  # Low Freq Ratio
    f4 = zcr / num_samples             
    f5 = peaks / num_samples           
    
    max_env = max(envelope)
    if max_env == 0: max_env = 1
    env_norm = [e / max_env for e in envelope] 
    
    return [f1, f2, f3, f4, f5] + env_norm

def extract_39_features(raw_buffer, noise_floor):
    start_idx, end_idx = get_trim_indices(raw_buffer, noise_floor)
    trimmed_len = end_idx - start_idx
    
    bytes_per_part = ((trimmed_len // 2) // 3) * 2
    
    p1_s = start_idx
    p1_e = p1_s + bytes_per_part
    p2_s = p1_e
    p2_e = p2_s + bytes_per_part
    p3_s = p2_e
    p3_e = end_idx 
    
    # Apply the new Smart Extractor to Start, Middle, and End of the word
    m1 = compute_smart_features(raw_buffer, p1_s, p1_e, 13)
    m2 = compute_smart_features(raw_buffer, p2_s, p2_e, 13)
    m3 = compute_smart_features(raw_buffer, p3_s, p3_e, 13)
    
    return m1 + m2 + m3

# ==========================================
# 2. DATASET PROCESSING
# ==========================================
def load_wav_as_bytes(filepath):
    try:
        with wave.open(filepath, 'rb') as w:
            if w.getframerate() != 16000 or w.getsampwidth() != 2:
                return None
            frames = w.readframes(w.getnframes())
            return bytearray(frames)
    except Exception:
        return None

X = []
y = []

print("Processing Target Words (with Pyboard Emulator)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir): continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"):
        raw_bytes = load_wav_as_bytes(os.path.join(word_dir, file_name))
        if raw_bytes and len(raw_bytes) > 4000:
            # We estimate Kaggle studio noise floor to be around 100
            features = extract_39_features(raw_bytes, noise_floor=100.0)
            X.append(features)
            y.append(label_idx)

print("\nProcessing Unknown Words...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir): continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word}"):
        raw_bytes = load_wav_as_bytes(os.path.join(word_dir, file_name))
        if raw_bytes and len(raw_bytes) > 4000:
            features = extract_39_features(raw_bytes, noise_floor=100.0)
            X.append(features)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nAwesome! Saved {len(X)} samples using Pyboard-native algorithms.")
print(f"X shape: {X.shape}")

Processing Target Words (with Pyboard Emulator)...


Loading stop: 100%|██████████| 1500/1500 [00:30<00:00, 48.63it/s]



Processing Unknown Words...


Loading down: 100%|██████████| 375/375 [00:07<00:00, 51.06it/s]


Awesome! Saved 6000 samples using Pyboard-native algorithms.
X shape: (6000, 39)


In [2]:
import numpy as np
import os
import struct

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import optuna

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load Data
X_raw = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y_raw = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

# --- NEW: Filter out 'off' class and remap labels ---
# Old labels: 0='on', 1='off', 2='stop', 3='unknown'
mask = (y_raw != 1) # Keep everything except 'off'
X = X_raw[mask]
y_filtered = y_raw[mask]

# Remap remaining labels to be sequential (0, 1, 2)
y = np.zeros_like(y_filtered)
y[y_filtered == 0] = 0 # 'on'
y[y_filtered == 2] = 1 # 'stop'
y[y_filtered == 3] = 2 # 'unknown'

print(f"Loaded & Filtered Data -> X shape: {X.shape}, y shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Standard Scaler
print("\n--- Applying StandardScaler ---")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. OPTUNA Optimization for Neural Network
print("\n--- Starting Optuna optimization for Neural Network ---")
def objective(trial):
    hidden_size = trial.suggest_int('hidden_size', 16, 32)
    alpha = trial.suggest_float('alpha', 1e-4, 1e-1, log=True)
    
    mlp = MLPClassifier(hidden_layer_sizes=(hidden_size,), activation='relu', 
                        solver='adam', alpha=alpha, max_iter=2000, random_state=42)
    mlp.fit(X_train_scaled, y_train)
    preds = mlp.predict(X_test_scaled)
    return accuracy_score(y_test, preds)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 4. Train Final Model
print("\n--- Training Final Lightweight Neural Network ---")
final_mlp = MLPClassifier(hidden_layer_sizes=(study.best_params['hidden_size'],), 
                          activation='relu', solver='adam', 
                          alpha=study.best_params['alpha'], max_iter=5000, random_state=42)
final_mlp.fit(X_train_scaled, y_train)
preds = final_mlp.predict(X_test_scaled)

# Updated target names (3 classes now)
target_names = ['on', 'stop', 'unknown']
print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. EXPORT NEURAL NETWORK AS BINARY
print("\n--- Exporting Neural Network as Binary ---")
output_file = os.path.join(OUTPUT_PATH, 'model_weights.bin')

with open(output_file, 'wb') as f:
    # Write network dimensions (Features=39, Hidden, Classes=3)
    f.write(struct.pack('<HHH', 39, study.best_params['hidden_size'], 3))
    
    # Write Scaler parameters
    for val in scaler.mean_: f.write(struct.pack('<f', float(val)))
    for val in scaler.scale_: f.write(struct.pack('<f', float(val)))
    
    # Write Hidden Layer (W1, B1)
    for row in final_mlp.coefs_[0]:
        for val in row: f.write(struct.pack('<f', float(val)))
    for val in final_mlp.intercepts_[0]:
        f.write(struct.pack('<f', float(val)))
        
    # Write Output Layer (W2, B2)
    for row in final_mlp.coefs_[1]:
        for val in row: f.write(struct.pack('<f', float(val)))
    for val in final_mlp.intercepts_[1]:
        f.write(struct.pack('<f', float(val)))

print(f"Neural Network successfully exported to: {output_file}")
print("!!! ACTION REQUIRED: Move 'model_weights.bin' to your SD Card !!!")

c:\Users\User\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loaded & Filtered Data -> X shape: (4500, 39), y shape: (4500,)

--- Applying StandardScaler ---

--- Starting Optuna optimization for Neural Network ---
Best Accuracy during CV: 82.00%

--- Training Final Lightweight Neural Network ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.78      0.83      0.80       300
        stop       0.92      0.87      0.89       300
     unknown       0.77      0.77      0.77       300

    accuracy                           0.82       900
   macro avg       0.82      0.82      0.82       900
weighted avg       0.82      0.82      0.82       900


--- Exporting Neural Network as Binary ---
Neural Network successfully exported to: ../edge_mcu\model_weights.bin
!!! ACTION REQUIRED: Move 'model_weights.bin' to your SD Card !!!
